# Job Postings Skill Demand — Topic 26
Synthetic data, January 2024–June 2025. This is not a live labour-market study.
The PDF generator is preserved in the source package. The executed version changes only
the `date_add` day-offset type to INT because Spark rejects BIGINT.
All business transformations below are generated from the same source tested locally.
Named tables and raw files are scoped to this capstone. Reruns replace this snapshot.


In [ ]:
import json, re, hashlib
from datetime import datetime, timezone
from pyspark.sql import functions as F
MY_ID = "23051560"
assert re.fullmatch(r"[A-Za-z0-9_]+", MY_ID)
SCHEMA = f"workspace.capstone_{MY_ID}"
PREFIX = f"sd_{MY_ID}_"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.raw")
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
def save(name, df):
    target = f"{SCHEMA}.{PREFIX}{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    result = spark.table(target)
    print(name, result.count())
    return result


## 1. Generate the specified source data

In [ ]:
FL = ("'Python','SQL','Spark','Airflow','Kafka','Hive','AWS','Azure','GCP','Docker',"
      "'Kubernetes','Terraform','Scala','Java','PowerBI','Tableau'")
SEC = "'BFSI','IT Services','Product','Retail','Health','Telecom','Energy','Logistics'"
CTY = ("'Bengaluru','Hyderabad','Pune','Chennai','Mumbai','Delhi NCR','Noida',"
       "'Kolkata','Ahmedabad','Kochi'")
P = lambda x: f"element_at(array({FL}), cast(pmod({x}, 16) as int) + 1)"
def wr(d, p):  # both whitespace flags off, or the padded rows lose their spaces
    (d.write.mode('overwrite').option('header', True)
     .option('ignoreLeadingWhiteSpace', False)
     .option('ignoreTrailingWhiteSpace', False).csv(f'{VOL}/raw/{p}'))
wr(spark.range(400).selectExpr("format_string('CMP%03d', id) company_id",
    "format_string('Company %03d', id + 1) company_name",
    f"element_at(array({SEC}), cast(id div 50 as int) + 1) sector",
    f"element_at(array({CTY}), cast(pmod(id, 10) as int) + 1) city"), 'companies')
b = spark.range(54000).selectExpr("id", "cast(id div 3000 as int) m",
    "cast(pmod(id, 3000) as int) k")
b = b.selectExpr("id", "m", "k", "pmod((m*2700 + k)*7919, 48600) ak",
    f"concat(array_distinct(array({P('id')}, {P('id div 16')}, {P('id div 256')},"
    f" {P('id div 4096')})), if(k < 520 + 22*m, array('Snowflake'), array()),"
    " if(k >= 1000 and k < 1400 + 18*m, array('Databricks'), array()),"
    " if(k >= 1800 and k < 2660 - 38*m, array('Hadoop'), array()),"
    " if(k < 520 + 22*m and pmod(k, 2) = 0, array('dbt'), array())) a")
b = b.selectExpr("k", "format_string('JP%06d', id) posting_id",
    "if(k >= 2775 and k < 2805, 'CMP999',"
    " format_string('CMP%03d', pmod(id, 400))) company_id",
    "if(k >= 2805 and k < 2820, '0000-00-00', date_format(date_add(add_months("
    "date'2024-01-01', m), cast(pmod(id, 28) as int)), 'yyyy-MM-dd')) posted_date",
    "case when k >= 2700 and k < 2750 then ''"
    " when k >= 2750 and k < 2775 then 'Not specified'"
    " when k < 2700 and ak < 1000 then array_join(a, ',')"
    " when k < 2700 and ak < 3200 then upper(array_join(a, ';'))"
    " when k < 2700 and ak < 4900 then concat(' ', array_join(a, ' ; '), ' ')"
    " else array_join(a, ';') end skills_raw")
wr(b.unionByName(b.filter('k >= 2820 and k < 2895')).drop('k'), 'postings')


## 2. Shared Bronze, Silver, Gold and verification logic

In [ ]:
"""Shared Spark transformations: identical business logic locally and on Databricks."""
from pyspark.sql import functions as F, Window

SOURCE_COLUMNS = {
    "companies": ["company_id", "company_name", "sector", "city"],
    "postings": ["posting_id", "company_id", "posted_date", "skills_raw"],
}


def bronze(spark, raw_root, save, cloud=False):
    result = {}
    for name, columns in SOURCE_COLUMNS.items():
        df = (spark.read.option("header", True).option("inferSchema", False)
              .option("ignoreLeadingWhiteSpace", False)
              .option("ignoreTrailingWhiteSpace", False).csv(
                  f"{raw_root}/{name}" if cloud else f"{raw_root}/{name}/part.csv"))
        if df.columns != columns:
            raise ValueError(f"Unexpected {name} header: {df.columns}")
        # Unity Catalog uses file metadata; local Apache Spark uses input_file_name.
        source = F.col("_metadata.file_path") if cloud else F.input_file_name()
        df = (df.withColumn("_source_file", source)
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_row_hash", F.sha2(F.to_json(F.struct(*columns),
                          options={"ignoreNullFields": "false"}), 256)))
        result[name] = save(f"bronze_{name}", df)
    return result


def silver(br, save):
    p = br["postings"]
    win = Window.partitionBy("posting_id").orderBy(F.desc("_ingested_at"), "_row_hash")
    ranked = p.withColumn("_rn", F.row_number().over(win))
    dupes = ranked.filter("_rn > 1").drop("_rn").withColumn("reject_reason", F.lit("duplicate_posting"))
    unique = ranked.filter("_rn = 1").drop("_rn")
    c = save("silver_companies", br["companies"].select(*SOURCE_COLUMNS["companies"]))
    known = c.select(F.col("company_id").alias("_known_company"))
    marked = (unique.join(known, unique.company_id == known._known_company, "left")
              .withColumn("_date", F.expr("try_cast(posted_date AS DATE)"))
              .withColumn("reject_reason",
                  F.when(F.col("skills_raw").isNull() | (F.trim("skills_raw") == "") |
                         (F.lower(F.trim("skills_raw")) == "not specified"), "no_skills_listed")
                   .when(F.col("_known_company").isNull(), "unknown_company")
                   .when(F.col("_date").isNull(), "unparseable_date")))
    rejected = marked.filter("reject_reason IS NOT NULL").select(*dupes.columns)
    rejects = save("silver_rejects", dupes.unionByName(rejected))
    accepted = (marked.filter("reject_reason IS NULL").drop("reject_reason", "_known_company", "posted_date")
                .withColumnRenamed("_date", "posted_date")
                .withColumn("month", F.trunc("posted_date", "month")))
    postings = save("silver_postings", accepted)
    tokens = (postings.select("posting_id", "company_id", "posted_date", "month",
                            F.explode(F.split("skills_raw", "[;,]")).alias("skill_raw")))
    skills = (tokens.withColumn("skill", F.lower(F.trim("skill_raw")))
              .filter("skill <> ''").dropDuplicates(["posting_id", "skill"])
              .join(c.select("company_id", "company_name", "sector", "city"), "company_id"))
    skills = save("silver_posting_skill", skills)
    return {"postings": postings, "skills": skills, "tokens": tokens, "rejects": rejects}


def gold(si, save):
    s = si["skills"]
    totals = si["postings"].groupBy("month").agg(F.count("*").alias("postings_that_month"))
    counts = s.groupBy("month", "skill").agg(F.countDistinct("posting_id").alias("postings_with_skill"))
    sm = (counts.join(totals, "month")
          .withColumn("skill_share", F.col("postings_with_skill") / F.col("postings_that_month"))
          .withColumn("rank_in_month", F.row_number().over(
              Window.partitionBy("month").orderBy(F.desc("postings_with_skill"), "skill"))))
    sm = save("gold_skill_month", sm.select("skill", "month", "postings_with_skill",
                     "postings_that_month", "skill_share", "rank_in_month"))
    a, b = s.select("posting_id", "month", "skill").alias("a"), s.select("posting_id", "month", "skill").alias("b")
    pair = (a.join(b, (F.col("a.posting_id") == F.col("b.posting_id")) &
                         (F.col("a.skill") != F.col("b.skill")))
            .select(F.col("a.month").alias("month"), F.col("a.skill").alias("skill_a"),
                    F.col("b.skill").alias("skill_b"))
            .groupBy("month", "skill_a", "skill_b").agg(F.count("*").alias("pair_postings")))
    ca = counts.select("month", F.col("skill").alias("skill_a"), F.col("postings_with_skill").alias("postings_a"))
    cb = counts.select("month", F.col("skill").alias("skill_b"), F.col("postings_with_skill").alias("postings_b"))
    pairs = (pair.join(ca, ["month", "skill_a"]).join(cb, ["month", "skill_b"]).join(totals, "month")
             .withColumn("confidence_a_to_b", F.col("pair_postings") / F.col("postings_a"))
             .withColumn("lift", F.col("pair_postings") * F.col("postings_that_month") /
                         (F.col("postings_a") * F.col("postings_b"))))
    pairs = save("gold_pair_month", pairs.select("skill_a", "skill_b", "month", "pair_postings",
                       "postings_a", "postings_b", "postings_that_month", "confidence_a_to_b", "lift"))
    return {"skill_month": sm, "pair_month": pairs}


def validate(spark, br, si, go):
    """Fail hard on structural errors; preserve discrepancies in the supplied brief."""
    checks = []
    def check(name, actual, expected, critical=True):
        passed = actual == expected
        checks.append(dict(check=name, actual=actual, expected=expected, passed=passed, critical=critical))
        print(f"{'PASS' if passed else 'FAIL'} {name}: {actual} (expected {expected})", flush=True)
    check("bronze_postings", br["postings"].count(), 55350)
    check("bronze_companies", br["companies"].count(), 400)
    check("silver_postings", si["postings"].count(), 51840)
    reasons = {r.reject_reason: r['count'] for r in si["rejects"].groupBy("reject_reason").count().collect()}
    for reason, n in [("duplicate_posting",1350),("no_skills_listed",1350),("unknown_company",540),("unparseable_date",270)]:
        check(reason, reasons.get(reason,0), n)
    check("row_reconciliation", si["postings"].count() + si["rejects"].count(), br["postings"].count())
    check("distinct_raw_skills", si["tokens"].select("skill_raw").distinct().count(), 57)
    check("canonical_skills", si["skills"].select("skill").distinct().count(), 20)
    check("invalid_clean_tokens", si["skills"].filter("skill <> lower(trim(skill)) OR skill LIKE '%,%' OR skill LIKE '%;%' OR skill = ''").count(),0)
    check("duplicate_posting_skill_keys", si["skills"].groupBy("posting_id","skill").count().filter("count > 1").count(),0)
    check("gold_skill_month", go["skill_month"].count(),360)
    check("gold_pair_month", go["pair_month"].count(),6660,False)
    check("monthly_denominators", sorted({r.postings_that_month for r in go["skill_month"].select("postings_that_month").distinct().collect()}), [2880])
    for skill, expected in [("snowflake",12726),("databricks",9954),("hadoop",9666),("dbt",6363)]:
        check(f"mentions_{skill}", si["skills"].filter(F.col("skill")==skill).count(),expected)
    for skill, first, last in [("snowflake",520,894),("databricks",400,706),("hadoop",860,214)]:
        rows = go["skill_month"].filter(F.col("skill")==skill).orderBy("month").collect()
        check(f"{skill}_first_last",[rows[0].postings_with_skill,rows[-1].postings_with_skill],[first,last])
    check("pair_bounds",go["pair_month"].filter("pair_postings > postings_a OR pair_postings > postings_b OR confidence_a_to_b > 1 OR lift < 0 OR skill_a = skill_b").count(),0)
    gp = go["pair_month"].groupBy("skill_a","skill_b").agg(F.sum("pair_postings").alias("n"),F.sum("postings_a").alias("d"))
    companions = gp.filter("skill_a = 'snowflake'").withColumn("confidence",F.col("n")/F.col("d")).orderBy(F.desc("confidence"),"skill_b").collect()
    check("snowflake_top_companion",companions[0].skill_b,"dbt",False)
    check("snowflake_dbt_confidence",next(r.confidence for r in companions if r.skill_b=="dbt"),0.5)
    check("other_companions_below_030",all(r.confidence<0.3 for r in companions if r.skill_b!="dbt"),True,False)
    failed = [c['check'] for c in checks if c['critical'] and not c['passed']]
    return checks, [r.asDict() for r in companions], failed


In [ ]:
br = bronze(spark, f"{VOL}/raw", save, cloud=True)
si = silver(br, save)
go = gold(si, save)
checks, companions, failures = validate(spark, br, si, go)
report = {"timestamp_utc": datetime.now(timezone.utc).isoformat(),
          "checks": checks, "companions": companions, "critical_failures": failures}
save("validation_results", spark.createDataFrame([(c["check"], str(c["actual"]), str(c["expected"]),
         c["passed"], c["critical"]) for c in checks],
         "check string, actual string, expected string, passed boolean, critical boolean"))
assert not failures, f"Pipeline correctness checks failed: {failures}"
print("CORE PIPELINE VALIDATED")
display(spark.table(f"{SCHEMA}.{PREFIX}validation_results"))


## 3. Export to the Volume for Snowflake staging

In [ ]:
# One header-bearing CSV per table; preserve the Silver table required by Q3.
exports = {
    "gold_skill_month": go["skill_month"].orderBy("month", "skill"),
    "gold_pair_month": go["pair_month"].orderBy("month", "skill_a", "skill_b"),
    "silver_posting_skill": si["skills"].select("posting_id", "company_id", "posted_date", "month",
             "skill", "company_name", "sector", "city").orderBy("posting_id", "skill"),
}
for name, df in exports.items():
    folder = f"{VOL}/exports/{name}_parts"
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(folder)
    parts = [x.path for x in dbutils.fs.ls(folder) if x.name.startswith("part-") and x.name.endswith(".csv")]
    assert len(parts) == 1, parts
    # Copy within this project's volume; no driver collection of the Silver export.
    destination = f"{VOL}/exports/{name}.csv"
    # Hadoop copy with overwrite scoped to this exact generated output.
    import shutil
    shutil.copyfile(parts[0].replace("dbfs:", ""), destination)
    with open(destination, "rb") as f:
        checksum = hashlib.file_digest(f, "sha256").hexdigest()
    print("EXPORT", destination, df.count(), checksum)
with open(f"{VOL}/exports/cloud_validation.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print("DATABRICKS PIPELINE AND EXPORT COMPLETE")


## Monthly top ten
The authoritative cross-platform queries are in `snowflake/02_analysis.sql`.

In [ ]:
display(spark.sql(f'-- Q1: monthly top ten. Deterministic alphabetical tie-break.\nSELECT skill, month, postings_with_skill, postings_that_month,\n       ROUND(100.0 * skill_share, 2) AS share_pct, rank_in_month\nFROM {SCHEMA}.{PREFIX}gold_skill_month\nQUALIFY ROW_NUMBER() OVER (PARTITION BY month ORDER BY postings_with_skill DESC, skill) <= 10\nORDER BY month, rank_in_month'))


## Quarterly growth
The authoritative cross-platform queries are in `snowflake/02_analysis.sql`.

In [ ]:
display(spark.sql(f"-- Q2: quarter-on-quarter change in share, in percentage points.\n-- The denominator is summed once per skill/month, never across all skills.\nWITH q AS (\n  SELECT skill, DATE_TRUNC('quarter', month) AS quarter,\n         SUM(postings_with_skill) AS mentions,\n         SUM(postings_that_month) AS postings,\n         100.0 * CAST(SUM(postings_with_skill) AS DOUBLE) / NULLIF(SUM(postings_that_month), 0) AS share_pct\n  FROM {SCHEMA}.{PREFIX}gold_skill_month GROUP BY skill, DATE_TRUNC('quarter', month)\n), growth AS (\n  SELECT *, share_pct - LAG(share_pct) OVER (PARTITION BY skill ORDER BY quarter) AS change_pp\n  FROM q\n)\nSELECT *, ROW_NUMBER() OVER (PARTITION BY quarter ORDER BY change_pp DESC, skill) AS growth_rank\nFROM growth\nORDER BY quarter, growth_rank"))


## Companion skills
The authoritative cross-platform queries are in `snowflake/02_analysis.sql`.

In [ ]:
display(spark.sql(f'-- Q3: top three companion skills from row-level Silver, matching PDF p107.\n-- Confidence is directional: P(B|A) is different from P(A|B).\nWITH s AS (SELECT DISTINCT posting_id, skill FROM {SCHEMA}.{PREFIX}silver_posting_skill),\n     n AS (SELECT skill, COUNT(*) AS c FROM s GROUP BY skill),\n     t AS (SELECT COUNT(DISTINCT posting_id) AS total FROM s),\n     pairs AS (\nSELECT a.skill AS skill_a, b.skill AS skill_b, COUNT(*) AS pair_postings,\n       ROUND(COUNT(*) / NULLIF(MAX(na.c), 0), 6) AS confidence_a_to_b,\n       ROUND(COUNT(*) * MAX(t.total) / NULLIF(MAX(na.c) * MAX(nb.c), 0), 6) AS lift\nFROM s a JOIN s b ON a.posting_id = b.posting_id AND a.skill <> b.skill\nJOIN n na ON na.skill = a.skill JOIN n nb ON nb.skill = b.skill CROSS JOIN t\nGROUP BY a.skill, b.skill\n)\nSELECT * FROM pairs\nQUALIFY ROW_NUMBER() OVER (PARTITION BY skill_a ORDER BY pair_postings DESC, skill_b) <= 3\nORDER BY skill_a, confidence_a_to_b DESC, skill_b'))
